<a href="https://colab.research.google.com/github/GitHub30/OpenJev/blob/main/notebooks/OpenJev_Quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenJev クイックスタート (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GitHub30/OpenJev/blob/main/notebooks/OpenJev_Quickstart.ipynb)

オープンウェイト LLM で動く System One モデル **OpenJev** を Colab で動かします。

1. GPU 確認 → 2. コード取得 → 3. インストール → 4. 推論 (noul / choice / score) → 5. 評価と校正 → 6. API サーバー + 公式 SDK → 7. (任意) LoRA 微調整

**ランタイム**: 「ランタイムのタイプを変更」で GPU を選んでください。A100 / L4 なら 7B、T4 なら 1.5B を自動で選びます。

In [1]:
!nvidia-smi
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
MODEL = "Qwen/Qwen2.5-7B-Instruct" if vram_gb > 20 else "Qwen/Qwen2.5-1.5B-Instruct"
print("VRAM: %.0f GB -> MODEL = %s" % (vram_gb, MODEL))

Sun Sep 20 02:18:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. コードを取得

[GitHub30/OpenJev](https://github.com/GitHub30/OpenJev) を clone します。ローカルの変更を試したい場合は `REPO_URL` を空にすると zip のアップロードに切り替わります。

In [2]:
import os, shutil, zipfile

REPO_URL = "https://github.com/GitHub30/OpenJev.git"  # 自分の fork を使う場合はここを変更

if os.path.isdir("/content/OpenJev"):
    print("already present")
elif REPO_URL:
    !git clone -q $REPO_URL /content/OpenJev
else:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall("/content/_upload")
    # zip の中の pyproject.toml がある階層を探す
    root = next(d for d, _, fs in os.walk("/content/_upload") if "pyproject.toml" in fs)
    shutil.move(root, "/content/OpenJev")

%cd /content/OpenJev
!ls

/content/OpenJev
examples  LICENSE  notebooks  pyproject.toml  README.md  scripts  src  tests


## 3. インストール

(ライブラリを `src/` から編集して試したい場合は `pip install -e .` の後に「ランタイム → セッションを再起動」してください)

In [3]:
# editable (-e) にすると .pth がカーネル再起動まで読まれず import に失敗するので通常インストールにする
%pip install -q ".[hf,server,dev]" peft typesafe-sdk
# Colab 同梱の torchao 0.10 は peft が要求する >=0.16 より古く、LoRA 作成時に ImportError になる。使わないので外す
%pip uninstall -q -y torchao
import importlib; importlib.invalidate_caches()
import openjev, transformers
print("openjev", openjev.__version__, "/ transformers", transformers.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
openjev 0.1.0 / transformers 5.16.1


## 4. 推論: 1 回の forward で複数の型付き質問に答える

テキストは生成しません。各質問の候補 (Yes/No、選択肢番号、レベル番号) の対数尤度を 1 バッチで採点し、確率分布に変換します。

In [4]:
import json, time
from openjev import SystemOneEngine
from openjev.backends.hf import HFBackend

backend = HFBackend(MODEL, dtype="bfloat16")   # 初回はモデルのダウンロードに数分かかります
engine = SystemOneEngine(backend=backend, model_name=f"openjev/{MODEL.split('/')[-1]}")

state = ("Hi, I've been trying to connect my Stripe account for 3 days and it keeps failing. "
         "I'm losing sales. Please help ASAP.")
questions = {
    "urgency": {"type": "noul", "instructions": "Does this message express urgency?"},
    "spanish": {"type": "noul", "instructions": "Is this message written in Spanish?"},
    "team": {
        "type": "choice",
        "instructions": "Which team should handle this ticket?",
        "criteria": {
            "billing": "Charges, refunds and invoices",
            "integrations": "Connecting third-party services such as Stripe or Slack",
            "account": "Login, password and profile problems",
            "other": None,
        },
    },
    "severity": {
        "type": "score",
        "instructions": "How severe is the customer's problem?",
        "criteria": ["Cosmetic issue", "Broken feature, but a workaround exists", "Blocking issue with no workaround"],
    },
}

t = time.time()
response = engine.system_one(state, questions)
print(f"{(time.time() - t) * 1000:.0f} ms\n")
print(json.dumps(response.model_dump(), indent=2))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

1051 ms

{
  "model": "openjev/Qwen2.5-7B-Instruct",
  "answers": {
    "urgency": {
      "type": "noul",
      "noul": 0.9999999715466358
    },
    "spanish": {
      "type": "noul",
      "noul": 8.423457729463323e-12
    },
    "team": {
      "type": "choice",
      "choice": "integrations",
      "confidence": 0.9999998778327275,
      "probabilities": {
        "billing": 9.056077508384593e-11,
        "integrations": 0.9999999380684693,
        "account": 1.6052281471294652e-09,
        "other": 6.023574182903244e-08
      }
    },
    "severity": {
      "type": "score",
      "score": 1.999999999751247,
      "confidence": 0.9999841463942563,
      "legend": {
        "0": "Cosmetic issue",
        "1": "Broken feature, but a workaround exists",
        "2": "Blocking issue with no workaround"
      },
      "probabilities": {
        "0": 1.2917800044663304e-12,
        "1": 2.4616969512062095e-10,
        "2": 0.9999999997525386
      }
    }
  },
  "usage": {
    "input_t

### 質問を増やしても時間はほぼ変わらない

state の KV キャッシュは 1 回だけ計算し、質問ごとの短い接尾辞だけを並列に採点します。

In [5]:
import time
for n in (1, 4, 16, 64):
    qs = {f"q{i}": {"type": "noul", "instructions": f"Does the message mention topic #{i} (sales, payments, urgency, Stripe, login...)?"} for i in range(n)}
    engine.system_one(state, qs)  # warm-up
    t = time.time(); engine.system_one(state, qs); dt = time.time() - t
    print(f"{n:3d} questions: {dt*1000:7.0f} ms  ({dt*1000/n:6.1f} ms / question)")

  1 questions:      74 ms  (  73.6 ms / question)
  4 questions:      89 ms  (  22.2 ms / question)
 16 questions:     165 ms  (  10.3 ms / question)
 64 questions:     503 ms  (   7.9 ms / question)


### confidence でルーティングする (confidence-gated routing)

In [6]:
team = response.answers["team"]
if team.confidence < 0.5:
    print("low confidence -> route to a human:", team.probabilities)
else:
    print(f"route to {team.choice} (confidence {team.confidence:.2f})")

route to integrations (confidence 1.00)


## 5. 評価と校正

`examples/triage.jsonl` (state / questions / gold の JSONL) で accuracy・NLL・ECE を測り、型ごとの温度を当てはめます。
自分のデータも同じ形式にすれば同じコマンドで使えます。

In [7]:
import math
from openjev.calibration import expected_calibration_error, fit_temperature, Calibration
from openjev.cli import _iter_jsonl, _gold_index
from openjev.schema import SystemOneRequest

def evaluate(engine, path):
    n = correct = 0; nll = 0.0; confs = []; oks = []
    raw_by_kind = {"noul": ([], []), "choice": ([], []), "score": ([], [])}
    for row in _iter_jsonl(path):
        req = SystemOneRequest.model_validate({"state": row["state"], "questions": row["questions"]})
        raws, _ = engine.evaluate_raw(req)
        for raw in raws:
            if raw.decision.name not in row["gold"]:
                continue
            g = _gold_index(raw.decision.kind, raw.decision.outcomes, row["gold"][raw.decision.name])
            p = raw.probabilities; pred = max(range(len(p)), key=p.__getitem__)
            n += 1; correct += pred == g; nll -= math.log(max(p[g], 1e-12))
            confs.append(p[pred]); oks.append(pred == g)
            raw_by_kind[raw.decision.kind][0].append(raw.logprobs); raw_by_kind[raw.decision.kind][1].append(g)
    print(f"n={n} accuracy={correct/n:.3f} nll={nll/n:.3f} ece={expected_calibration_error(confs, oks):.3f}")
    return raw_by_kind

print("before calibration:")
raw_by_kind = evaluate(engine, "examples/triage.jsonl")

cal = Calibration()
for kind, (sets, gold) in raw_by_kind.items():
    if sets:
        cal.temperatures[kind] = fit_temperature(sets, gold)
print("temperatures:", cal.temperatures)
engine.calibration = cal
print("after calibration:")
evaluate(engine, "examples/triage.jsonl")
cal.save("calibration.json")

before calibration:
n=48 accuracy=0.833 nll=1.555 ece=0.182
temperatures: {'noul': 3.6921946659168223, 'choice': 6.049249893134713, 'score': 5.943045572594548}
after calibration:
n=48 accuracy=0.833 nll=0.473 ece=0.120


## 6. API サーバーを立てて公式 `typesafe-sdk` から呼ぶ

`POST /v1/systemone` は TypeSafe の OpenAPI と同じ形式なので、SDK は `TYPESAFE_BASE_URL` を向けるだけで動きます。
(すでにロード済みのモデルを再利用するため、ノートブック内でバックグラウンドスレッドとして起動します)

In [8]:
import threading, time, uvicorn, requests
from openjev.server import create_app

app = create_app(engine)
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(50):
    try:
        print(requests.get("http://127.0.0.1:8000/healthz").json()); break
    except Exception:
        time.sleep(0.2)

{'status': 'ok', 'model': 'openjev/Qwen2.5-7B-Instruct', 'backend': 'Qwen/Qwen2.5-7B-Instruct'}


In [9]:
import os
os.environ["TYPESAFE_API_KEY"] = "anything"
os.environ["TYPESAFE_BASE_URL"] = "http://127.0.0.1:8000"

from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

with TypeSafeClient() as client:
    r = client.system_one(
        "I was charged twice for my subscription this month. Can you refund the duplicate charge?",
        {
            "billing": Noul(instructions="Is this about billing?"),
            "tone": Choice(instructions="What is the tone?", criteria={"calm": None, "angry": None}),
            "urgency": Score(instructions="How urgent is this?", criteria=["low", "medium", "high"]),
        },
    )
print("noul  :", r.nouls["billing"].noul)
print("choice:", r.choices["tone"].choice, r.choices["tone"].confidence)
print("score :", r.scores["urgency"].score, r.scores["urgency"].probabilities)
print("usage :", r.usage)

noul  : 0.9819237361642256
choice: angry 0.082467163455887
score : 0.9490199805623136 {0: 0.2956062409050521, 1: 0.45976753762758227, 2: 0.24462622146736565}
usage : input_tokens=354 output_tokens=7


In [10]:
!curl -s -X POST http://127.0.0.1:8000/v1/systemone \
  -H "Authorization: Bearer anything" -H "Content-Type: application/json" \
  -d '{"state": "Hola, necesito ayuda con mi factura.", "model": "jev-latest", "questions": {"spanish": {"type": "noul", "instructions": "Is this written in Spanish?"}}}' | python -m json.tool

{
    "model": "openjev/Qwen2.5-7B-Instruct",
    "answers": {
        "spanish": {
            "type": "noul",
            "noul": 0.9874737193730624
        }
    },
    "usage": {
        "input_tokens": 102,
        "output_tokens": 2
    }
}


## 7. (任意) LoRA で校正付き微調整

候補分布の NLL (+ Brier) を最小化する proper scoring rule 学習です。ここでは動作確認として `examples/triage.jsonl` (16 行) で数ステップ回すだけなので、
精度向上は期待しないでください。本番では数千件のデータを用意します (boolq / banking77 / clinc_oos / sst5 など)。

学習後は `HFBackend(MODEL, adapter="checkpoints/lora")` でアダプタをマージして使えます。
Colab は切断されるとファイルが消えるので、本番では `--output` を Google Drive のパスにしてください。

In [11]:
# GPU メモリを空けてから学習 (推論用モデルと学習用モデルを同時に載せないため)
server.should_exit = True
del engine, backend
import gc, torch; gc.collect(); torch.cuda.empty_cache()

!python scripts/train_calibrated.py --model $MODEL --data examples/triage.jsonl --eval-data examples/triage.jsonl \
    --output checkpoints/lora --grad-accum 8 --max-steps 6 --lora-r 8 --brier-weight 0.5

Loading weights: 100% 339/339 [00:00<00:00, 3607.68it/s]
trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643
48 training decisions
epoch 0 step 6/6 loss 1.9156
eval: n=48 accuracy=0.812 nll=1.5134
saved LoRA adapter to checkpoints/lora


In [12]:
from openjev import SystemOneEngine
from openjev.backends.hf import HFBackend

tuned = SystemOneEngine(backend=HFBackend(MODEL, dtype="bfloat16", adapter="checkpoints/lora"))
print(tuned.system_one(state, {"team": questions["team"]}).answers["team"])

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

type='choice' choice='integrations' confidence=0.9999997055372415 probabilities={'billing': 1.1032559984123996e-09, 'integrations': 0.9999998500352617, 'account': 4.363462118743383e-09, 'other': 1.444980201668203e-07}


## 次のステップ

- 自分のデータを `{"state": ..., "questions": {...}, "gold": {...}}` の JSONL にして `openjev eval` / `calibrate` / `train_calibrated.py` に渡す
- `criteria` に具体的な説明 (`what` / `not_for` / `examples`) を書くと選択肢の取り違えが減る
- `other` / 「該当なし」の選択肢を必ず入れる